In [ ]:
import pandas as pd
import altair as alt
alt.data_transformers.disable_max_rows()

url = "https://github.com/UIUC-iSchool-DataViz/is445_data/raw/main/ufo-scrubbed-geocoded-time-standardized-00.csv"


cols = [
    'datetime', 'city', 'state', 'country', 'shape',
    'duration_seconds', 'duration', 'comments',
    'date_posted', 'latitude', 'longitude'
]

df = pd.read_csv(url, header=None, names=cols)

df.head()

In [2]:
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')

In [ ]:
df = df.dropna(subset=['shape', 'country'])
df

In [ ]:
chart1 = alt.Chart(df).mark_line().encode(
    x='year(datetime):T',
    y='count()'
).properties(
    title='UFO Sightings Over Time'
)

chart1

In [5]:

df_us = df[df['country'] == 'us'].copy()

df_us['latitude'] = pd.to_numeric(df_us['latitude'], errors='coerce')
df_us['longitude'] = pd.to_numeric(df_us['longitude'], errors='coerce')

df_us = df_us.dropna(subset=['latitude', 'longitude', 'shape'])

In [6]:
states = alt.topo_feature(
    'https://cdn.jsdelivr.net/npm/vega-datasets@v1.29.0/data/us-10m.json',
    'states'
)

background = alt.Chart(states).mark_geoshape(
    fill='lightgray',
    stroke='white'
).properties(
    width=600,
    height=400
).project(
    type='albersUsa'
)

In [7]:
shape_list = sorted(df_us['shape'].unique())

select_shape = alt.selection_point(
    fields=['shape'],
    bind=alt.binding_select(
        options=shape_list,
        name='Shape: '
    )
)

In [8]:
points = alt.Chart(df_us).mark_circle(size=40).encode(
    longitude='longitude:Q',
    latitude='latitude:Q',
    
    color=alt.Color('shape:N'),
    
    opacity=alt.condition(select_shape, alt.value(1), alt.value(0.001)),
    
    tooltip=[
        'city:N',
        'state:N',
        'shape:N',
        'datetime:T'
    ]
)

In [ ]:
chart2 = (background + points).add_params(
    select_shape
).properties(
    title='UFO Sightings in the United States (Interactive Map)'
)

chart2

In [10]:
chart1.save('chart1.html')
chart2.save('chart2.html')